# Camada Gold

## Carregando dados das camadas Silver e Bronze

In [0]:
from pyspark.sql.types import IntegerType, DoubleType

silver_base = "/mnt/datalakef085704a8c687020/silver"
bronze_base = "/mnt/datalakef085704a8c687020/bronze"

# Dimensões Silver
df_cliente = spark.read.format("delta").load(f"{silver_base}/dim_cliente")
df_produto = spark.read.format("delta").load(f"{silver_base}/dim_produto")
df_promocao = spark.read.format("delta").load(f"{silver_base}/dim_promocao")
df_endereco = spark.read.format("delta").load(f"{silver_base}/dim_endereco")
df_categoria = spark.read.format("delta").load(f"{silver_base}/dim_categoria")
df_fornecedor = spark.read.format("delta").load(f"{silver_base}/dim_fornecedor")

# Bronze - fatos base para a fato_vendas
df_order = spark.read.format("delta").load(f"{bronze_base}/order")
df_order_item = spark.read.format("delta").load(f"{bronze_base}/order_item")
df_product_promotion = spark.read.format("delta").load(f"{bronze_base}/product_promotion")
df_payment = spark.read.format("delta").load(f"{bronze_base}/payment")

# Conversão de tipos básicos para evitar erro downstream
df_order = df_order.select(
    "order_id", "customer_id", "delivery_address_id", "order_date", "status", "total_amount"
).withColumn("order_id", df_order["order_id"].cast(IntegerType())) \
 .withColumn("customer_id", df_order["customer_id"].cast(IntegerType())) \
 .withColumn("delivery_address_id", df_order["delivery_address_id"].cast(IntegerType())) \
 .withColumn("total_amount", df_order["total_amount"].cast(DoubleType()))

df_order_item = df_order_item.select(
    "order_item_id", "order_id", "product_id", "quantity", "unit_price"
).withColumn("order_item_id", df_order_item["order_item_id"].cast(IntegerType())) \
 .withColumn("order_id", df_order_item["order_id"].cast(IntegerType())) \
 .withColumn("product_id", df_order_item["product_id"].cast(IntegerType())) \
 .withColumn("quantity", df_order_item["quantity"].cast(IntegerType())) \
 .withColumn("unit_price", df_order_item["unit_price"].cast(DoubleType()))


## Montando `fato_vendas` e aplicando descontos

In [0]:
from pyspark.sql.functions import col, when, current_timestamp

# Filtra promoções válidas (não expiradas)
df_promocao_valid = df_promocao.filter(col("expiration_date") > current_timestamp())

# Une produtos com promoções (se houver)
df_order_item_promo = df_order_item.join(df_product_promotion, "product_id", "left") \
    .join(df_promocao_valid, "promotion_id", "left") \
    .withColumn("discount_pct", when(col("discount_percentage").isNull(), 0).otherwise(col("discount_percentage")))

# Calcula preço total com desconto
df_order_item_promo = df_order_item_promo.withColumn(
    "total_price",
    col("quantity") * col("unit_price") * (1 - col("discount_pct") / 100)
)

# Monta fato_vendas com joins para enriquecer com dimensões
df_fato_vendas = df_order_item_promo \
    .join(df_order, "order_id") \
    .join(df_cliente.select("customer_id", "full_name"), "customer_id") \
    .join(df_produto.select("product_id", "product_name", "category_id", "supplier_id"), "product_id") \
    .join(df_categoria.select("category_id", "category_name"), "category_id") \
    .join(df_fornecedor.select("supplier_id", "supplier_name"), "supplier_id") \
    .join(df_endereco.select("address_id", "city", "state"), df_order["delivery_address_id"] == df_endereco["address_id"]) \
    .select(
        col("order_id"),
        col("order_date"),
        col("customer_id"),
        col("full_name").alias("customer_name"),
        col("product_id"),
        col("product_name"),
        col("category_id"),
        col("category_name"),
        col("supplier_id"),
        col("supplier_name"),
        col("delivery_address_id"),
        col("city").alias("delivery_city"),
        col("state").alias("delivery_state"),
        col("quantity"),
        col("unit_price"),
        col("discount_pct"),
        col("total_price"),
        col("status"),
        col("total_amount")
    )

## Criando dimensão tempo (`dim_tempo`)

In [0]:
from pyspark.sql.functions import current_timestamp

data_inicial = "2023-01-01"
data_final = "2026-12-31"

num_dias = spark.sql(f"SELECT datediff('{data_final}', '{data_inicial}')").collect()[0][0]

df_calendario = spark.range(0, num_dias + 1) \
    .selectExpr(f"date_add(to_date('{data_inicial}'), CAST(id AS INT)) AS data")

df_dim_tempo = df_calendario.selectExpr(
    "data",
    "year(data) AS ano",
    "month(data) AS mes",
    """(CASE month(data)
        WHEN 1 THEN 'JANEIRO'
        WHEN 2 THEN 'FEVEREIRO'
        WHEN 3 THEN 'MARCO'
        WHEN 4 THEN 'ABRIL'
        WHEN 5 THEN 'MAIO'
        WHEN 6 THEN 'JUNHO'
        WHEN 7 THEN 'JULHO'
        WHEN 8 THEN 'AGOSTO'
        WHEN 9 THEN 'SETEMBRO'
        WHEN 10 THEN 'OUTUBRO'
        WHEN 11 THEN 'NOVEMBRO'
        WHEN 12 THEN 'DEZEMBRO'
    END) AS nome_mes""",
    "day(data) AS dia",
    """(CASE dayofweek(data)
        WHEN 1 THEN 'DOMINGO'
        WHEN 2 THEN 'SEGUNDA-FEIRA'
        WHEN 3 THEN 'TERCA-FEIRA'
        WHEN 4 THEN 'QUARTA-FEIRA'
        WHEN 5 THEN 'QUINTA-FEIRA'
        WHEN 6 THEN 'SEXTA-FEIRA'
        WHEN 7 THEN 'SABADO'
    END) AS nome_dia_semana""",
    "dayofweek(data) AS numero_dia_semana"
)

gold_base = "/mnt/datalakef085704a8c687020/gold"
df_dim_tempo.write.format("delta").mode("overwrite").save(f"{gold_base}/dim_tempo")

print("Dimensão tempo criada e salva com sucesso!")

## Salvando `fato_vendas` na Gold

In [0]:
gold_base = "/mnt/datalakef085704a8c687020/gold"

df_fato_vendas.write.format("delta").mode("overwrite").save(f"{gold_base}/fato_vendas")

print("Tabela fato_vendas criada e salva com sucesso!")

## Calculando KPIs

In [0]:
from pyspark.sql.functions import sum as _sum, avg, countDistinct

gold_base = "/mnt/datalakef085704a8c687020/gold"

df_fato_vendas = spark.read.format("delta").load(f"{gold_base}/fato_vendas")

# KPI 1: Total vendas (R$)
total_vendas = df_fato_vendas.select(_sum("total_price")).collect()[0][0]

# KPI 2: Quantidade total vendida
quantidade_vendida = df_fato_vendas.select(_sum("quantity")).collect()[0][0]

# KPI 3: Ticket médio por pedido
numero_pedidos = df_fato_vendas.select(countDistinct("order_id")).collect()[0][0]
ticket_medio = total_vendas / numero_pedidos if numero_pedidos else 0

# KPI 4: Desconto médio aplicado (%)
desconto_medio = df_fato_vendas.select(avg("discount_pct")).collect()[0][0]

print(f"Total vendas: R$ {total_vendas:.2f}")
print(f"Quantidade total vendida: {quantidade_vendida}")
print(f"Ticket médio por pedido: R$ {ticket_medio:.2f}")
print(f"Desconto médio aplicado: {desconto_medio:.2f}%")


## Consultas de validação

In [0]:
gold_base = "/mnt/datalakef085704a8c687020/gold"

print("Mostrando a dimensão clientes:")
spark.read.format("delta").load(f"{gold_base}/dim_cliente").show(10, truncate=False)

print("Mostrando a dimensão produtos:")
spark.read.format("delta").load(f"{gold_base}/dim_produto").show(10, truncate=False)

print("Mostrando a dimensão categoria:")
spark.read.format("delta").load(f"{gold_base}/dim_categoria").show(10, truncate=False)

print("Mostrando a dimensão fornecedor:")
spark.read.format("delta").load(f"{gold_base}/dim_fornecedor").show(10, truncate=False)

print("Mostrando a dimensão endereço:")
spark.read.format("delta").load(f"{gold_base}/dim_endereco").show(10, truncate=False)

print("Mostrando a dimensão tempo:")
spark.read.format("delta").load(f"{gold_base}/dim_tempo").show(10, truncate=False)

print("Mostrando a tabela fato vendas:")
spark.read.format("delta").load(f"{gold_base}/fato_vendas").show(10, truncate=False)